# Notebook 10: Protocol Factorial and CPU Pre-GPU Screen

**Purpose**: Define demonstration protocols as a `mechanism x composition`
factorial (replacing v1's flat list of six named protocols), and screen the grid
on CPU before spending GPU time.

## Why the v1 protocol list was replaced

The v1 suite is six mutually exclusive arms. Two problems:

1. **They confound *what gets selected* with *the label composition of what gets
   selected*.** `label_diversity` is literally "random, forced to k/2 per class".
   `counter_spurious` over-samples minority cells, which are label-skewed by
   construction, so it silently ships a different label composition too. An
   arm-vs-arm comparison therefore cannot tell you whether an effect came from
   the selection mechanism or from the label counts.
2. **Pinning everything to k/2 per class** (`src/selection/balanced_topk.py`)
   fixes problem 1 by removing the channel entirely — but Notebook 09's
   reference table shows ID→OOD accuracy collapses where the label prior moves
   while AUROC stays flat, so label composition looked like the highest-leverage
   knob available, and a 50/50 pin is the one setting guaranteed not to use it.

So composition is promoted to an explicit crossed factor in
`src/selection/protocols.py`. The v1 arms become *cells* of the grid
(`label_diversity == random x balanced`, `similarity == similarity x free`), the
confound becomes an estimable interaction, and prior matching becomes testable
rather than assumed away.

> **Result from later notebooks, recorded here so this design note is not
> misleading:** the composition factor turned out *not* to matter. Notebook 11
> shows `balanced` demonstrations still produce a ~100% positive output rate at
> 8B, and Notebook 13 finds no systematic composition effect. The factorial was
> still the right design — it is what let us *measure* that rather than assume
> it — but Notebook 14 holds composition fixed and spends the budget on seeds
> instead.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## The grid

Seven mechanisms x three compositions. Every mechanism returns *pool index
labels*, not positional offsets — v1's `similarity_select.select` returned
positions into `pool_texts` while every other selector returned index labels,
an asymmetry that is a bug magnet at the call site and is not reproduced here.

Shift-aware mechanisms degrade gracefully: when the domain discriminator cannot
separate the domains (`DomainShift.detectable` is False — `anes`, per Notebook
09), the density ratio is noise, so `importance_weighted` and
`shift_axis_coverage` fall back to uniform behaviour and set `degraded=True` in
the returned metadata rather than sampling on noise.

In [2]:
from src.selection.protocols import (
    COMPOSITIONS, MECHANISMS, V1_EQUIVALENTS, protocol_grid,
)

print("mechanisms: ", MECHANISMS)
print("compositions:", COMPOSITIONS)
print(f"\ngrid cells: {len(protocol_grid())}")
print("\nv1 arms recovered as cells:")
for v1, cell in V1_EQUIVALENTS.items():
    print(f"  {v1:18s} == {cell[0]} x {cell[1]}")

mechanisms:  ('random', 'similarity', 'feature_coverage', 'rule_coverage', 'counter_spurious', 'importance_weighted', 'shift_axis_coverage')
compositions: ('free', 'balanced', 'target_prior')

grid cells: 20

v1 arms recovered as cells:
  random             == random x free
  label_diversity    == random x balanced
  feature_range      == feature_coverage x free
  rule_diversity     == rule_coverage x free
  similarity         == similarity x free
  counter_spurious   == counter_spurious x free


## What the screen measures, and what it deliberately does not

An LLM prompt at k=8 can only convey what is in the eight rows. This screen asks
the question that *precedes* "does the LLM do better": **is the information even
present in the demonstration set?** For every cell it builds the demo set the
protocol would put in the prompt and scores it with three order-invariant
surrogate readers, each standing in for a documented ICL behaviour:

| surrogate | behaviour it stands for | uses the query? | uses the input-label pairing? |
|---|---|---|---|
| `prior_only` | majority-label / label-copy bias (Zhao et al. 2021; the "label space, not input-label mapping" reading of Min et al. 2022) | no | no |
| `nn1` | retrieval — predict the nearest demo's label | yes | yes |
| `logreg` | task *learning* — L2 logistic regression fit on the eight rows | yes | yes |

The spread between `prior_only` and `logreg` is the headroom that demonstration
*content* offers over demonstration *label counts*. A cell where the two are
equal is one where no LLM, however good, can distinguish the protocol from its
label composition.

**Limits, stated plainly.** These surrogates are order-invariant, so the screen
says nothing about demo ordering, serialisation, or prompt format — the axes that
need the actual LLM. It is a *necessary-condition filter*, not a substitute for
the GPU runs: a protocol whose demo sets carry no more OOD information than
random-k is not worth GPU hours, but passing the screen does not imply an LLM
will exploit it. Notebook 11 shows exactly this — the `prior_only` surrogate's
core assumption (that demo label counts drive the output prior) turned out to be
**false** for the real model.

## Step 1: Run the screen

```bash
cd sata-project
PYTHONPATH=. python scripts/screen_protocols.py --cache-dir _screen_cache \
    --out results/v2 --n-queries 250 --seeds 42 123 456 789 1024
```

CPU only, a few minutes. The cell below loads the cached output.

In [3]:
import numpy as np
import pandas as pd

RESULTS = PROJECT_ROOT / "results" / "v2"
screen = pd.read_parquet(RESULTS / "protocol_screen.parquet")
ood = screen[screen.query_split == "ood"]
M = "balanced_accuracy"

print(f"{len(screen)} rows | seeds={sorted(screen.seed.unique())} | k={screen.k.unique()}")
screen.head(3)

2400 rows | seeds=[np.int64(42), np.int64(123), np.int64(456), np.int64(789), np.int64(1024)] | k=[8]


,dataset,mechanism,composition,seed,query_split,surrogate,k,query_conditional,degraded,accuracy,balanced_accuracy,pred_pos_rate,query_pos_rate,auroc,demo_pos_frac,demo_target_likeness,demo_leaf_coverage,demo_feature_spread,demo_query_dist
0,brfss_diabetes,random,free,42,ood,prior_only,8,False,False,0.788,0.505303,0.124,0.12,0.500000,0.125,0.469374,2.0,0.71766,0.443332
1,brfss_diabetes,random,free,42,ood,nn1,8,False,False,0.812,0.475758,0.076,0.12,0.432424,0.125,0.469374,2.0,0.71766,0.443332
2,brfss_diabetes,random,free,42,ood,logreg,8,False,False,0.856,0.500758,0.032,0.12,0.609394,0.125,0.469374,2.0,0.71766,0.443332


## Step 2: Why balanced accuracy, not raw accuracy

Raw accuracy is not a safe headline here. On `brfss_diabetes` the positive rate
is ~12.5%, so a demo set that simply carries the source label prior makes the
surrogate predict the majority class and score ~0.75 **without using the query at
all**. That would credit the `free` composition for class imbalance rather than
for conveying anything.

The cell below shows the artefact directly: on raw accuracy the composition
factor looks decisive; on balanced accuracy and AUROC — metrics it cannot game —
the effect largely disappears.

In [4]:
cmp_ = ood.groupby(["surrogate", "composition"])[[M, "accuracy", "auroc"]].mean().round(3)
cmp_

balanced_accuracy  accuracy  auroc
surrogate  composition                                     
logreg     balanced                  0.550     0.546  0.572
           free                      0.540     0.617  0.590
           target_prior              0.529     0.582  0.571
nn1        balanced                  0.571     0.562  0.630
           free                      0.569     0.624  0.631
           target_prior              0.562     0.585  0.618
prior_only balanced                  0.490     0.490  0.500
           free                      0.514     0.576  0.536
           target_prior              0.494     0.532  0.500

## Step 3: Mechanism effects and headroom

In [5]:
mech = ood[ood.surrogate == "logreg"].groupby(["dataset", "mechanism"])[M].mean().unstack().round(3)
mech

mechanism,counter_spurious,feature_coverage,importance_weighted,random,rule_coverage,shift_axis_coverage,similarity
dataset,,,,,,,
acsincome,0.407,0.583,0.618,0.578,0.579,0.579,0.642
acspubcov,0.445,0.504,0.499,0.496,0.501,0.495,0.530
anes,0.495,0.525,0.538,0.538,0.537,0.538,0.629
brfss_diabetes,0.420,0.596,0.539,0.555,0.514,0.550,0.543


In [6]:
head = ood.groupby(["dataset", "surrogate"])[M].mean().unstack().round(3)
head["headroom_logreg_minus_prior"] = (head["logreg"] - head["prior_only"]).round(3)
head

surrogate,logreg,nn1,prior_only,headroom_logreg_minus_prior
dataset,,,,
acsincome,0.577,0.591,0.501,0.076
acspubcov,0.498,0.512,0.496,0.002
anes,0.545,0.601,0.496,0.049
brfss_diabetes,0.537,0.565,0.503,0.034


**`acspubcov` has no headroom.** Every surrogate sits at chance, which predicts
that no demonstration protocol can work on that dataset. Notebooks 11 and 13
confirm this with the actual LLM at both 8B and 70B, so it is demoted to a
documented negative control from here on.

## Step 4: Ranking every cell against the random-k baseline

In [7]:
rows = []
for ds, g in ood[ood.surrogate == "logreg"].groupby("dataset"):
    base = g[(g.mechanism == "random") & (g.composition == "free")][M].mean()
    cell = g.groupby(["mechanism", "composition"])[M].mean()
    for (mech_, comp_), v in cell.items():
        rows.append(dict(dataset=ds, mechanism=mech_, composition=comp_,
                         balanced_accuracy=v, random_k_baseline=base,
                         delta_vs_random_k=v - base))
rank = pd.DataFrame(rows).sort_values(["dataset", "delta_vs_random_k"], ascending=[True, False])

for ds, g in rank.groupby("dataset"):
    print(f"{ds}  (random-k = {g.random_k_baseline.iloc[0]:.3f})")
    print(g.head(3)[["mechanism", "composition", "balanced_accuracy", "delta_vs_random_k"]]
            .round(3).to_string(index=False))
    print("  worst:", g.tail(1)[["mechanism", "composition", "delta_vs_random_k"]]
            .round(3).to_string(index=False, header=False), end="\n\n")

acsincome  (random-k = 0.552)
          mechanism composition  balanced_accuracy  delta_vs_random_k
         similarity        free              0.701              0.149
importance_weighted    balanced              0.668              0.116
         similarity    balanced              0.637              0.085
  worst: counter_spurious target_prior -0.146

acspubcov  (random-k = 0.503)
 mechanism  composition  balanced_accuracy  delta_vs_random_k
similarity     balanced              0.543               0.04
similarity         free              0.523               0.02
similarity target_prior              0.523               0.02
  worst: counter_spurious target_prior -0.071

anes  (random-k = 0.514)
 mechanism  composition  balanced_accuracy  delta_vs_random_k
similarity         free              0.663              0.149
similarity     balanced              0.626              0.112
similarity target_prior              0.599              0.085
  worst: counter_spurious balanced -0.028

br

## What carried forward to the GPU runs

1. `counter_spurious` falls **below chance** on all four datasets — carried
   forward as a single documented negative control, not a full row of the grid.
2. `similarity` leads on three of four. Query-conditional selection carries
   signal even for `prior_only`, which never learns a mapping — it turns the
   majority-label bias into an implicit k-NN classifier. This is the observation
   that eventually became Notebook 14's feature-channel hypothesis.
3. `acspubcov` has no headroom for any protocol.

**Five seeds cannot support inferential claims** about protocol differences —
see Notebook 14 for the arithmetic on why, and why later runs use eight.